In [ ]:
```python
import pandas as pd
import os

# =========================================================
# 1. DATASET LOCATION
# =========================================================

folder = "datasets"

# =========================================================
# 2. DATASETS
# =========================================================

files = {
    "Renewable": "Renewable Energy monitoring.xlsx",
    "SCADA": "scada_pipeline.csv",
    "Smart Meter": "smart_meter_data.csv",
    "Maintenance": "asset_maintenance.csv",
    "Billing": "Billing_payments.csv",
    "IIoT": "iiot_smart_grid_dataset.csv",
    "OWID": "owid-energy-data.json",
    "OMS": "OMS_data.csv"
}

# =========================================================
# 3. LOAD DATASETS
# =========================================================

def load_file(filename):
    path = os.path.join(folder, filename)

    if filename.endswith(".csv"):
        return pd.read_csv(path)
    elif filename.endswith(".xlsx"):
        return pd.read_excel(path, sheet_name="data")
    elif filename.endswith(".json"):
        return pd.read_json(path)

data = {}

for name, file in files.items():
    try:
        data[name] = load_file(file)
        print(f"{name} loaded successfully")
    except Exception as e:
        print(f"Error loading {name}: {e}")

# =========================================================
# 4. QUALITY REPORT
# =========================================================

quality = []

def add_issue(dataset, column, issue, count, severity):

    quality.append([
        dataset,
        column,
        issue,
        count,
        severity
    ])

# =========================================================
# 5. GENERAL QUALITY CHECKS
# =========================================================

for dataset, df in data.items():

    # Missing values
    for column in df.columns:

        count = df[column].isna().sum()

        if count > 0:

            severity = (
                "High" if count / len(df) > 0.10
                else "Medium"
            )

            add_issue(
                dataset,
                column,
                "Missing Values",
                count,
                severity
            )

    # Duplicate rows
    count = df.duplicated().sum()

    if count > 0:

        add_issue(
            dataset,
            "All Columns",
            "Duplicate Rows",
            count,
            "High"
        )

# =========================================================
# 6. SCADA QUALITY
# =========================================================

if "SCADA" in data:

    df = data["SCADA"]

    columns = [
        "pressure",
        "flow_rate",
        "temperature",
        "pump_speed",
        "energy_consumption"
    ]

    for column in columns:

        if column in df.columns:

            count = (df[column] < 0).sum()

            if count > 0:

                add_issue(
                    "SCADA",
                    column,
                    "Negative Sensor Value",
                    count,
                    "High"
                )

    if "timestamp" in df.columns:

        dates = pd.to_datetime(
            df["timestamp"],
            errors="coerce"
        )

        count = (
            dates.isna() &
            df["timestamp"].notna()
        ).sum()

        if count > 0:

            add_issue(
                "SCADA",
                "timestamp",
                "Invalid Timestamp",
                count,
                "High"
            )

# =========================================================
# 7. SMART METER QUALITY
# =========================================================

if "Smart Meter" in data:

    df = data["Smart Meter"]

    columns = [
        "Electricity_Consumed",
        "Wind_Speed",
        "Avg_Past_Consumption"
    ]

    for column in columns:

        if column in df.columns:

            count = (df[column] < 0).sum()

            if count > 0:

                add_issue(
                    "Smart Meter",
                    column,
                    "Negative Consumption Value",
                    count,
                    "High"
                )

    if "Humidity" in df.columns:

        count = (
            (df["Humidity"] < 0) |
            (df["Humidity"] > 100)
        ).sum()

        if count > 0:

            add_issue(
                "Smart Meter",
                "Humidity",
                "Humidity Outside 0-100 Range",
                count,
                "Medium"
            )

# =========================================================
# 8. IIoT QUALITY
# =========================================================

if "IIoT" in data:

    df = data["IIoT"]

    columns = [
        "Power_Consumption_kWh",
        "Reactive_Power_kVAR",
        "Active_Power_kW",
        "Solar_Power_Generation_kW",
        "Wind_Power_Generation_kW"
    ]

    for column in columns:

        if column in df.columns:

            count = (df[column] < 0).sum()

            if count > 0:

                add_issue(
                    "IIoT",
                    column,
                    "Negative Power/Energy Value",
                    count,
                    "High"
                )

    if "Power_Factor" in df.columns:

        count = (
            (df["Power_Factor"] < -1) |
            (df["Power_Factor"] > 1)
        ).sum()

        if count > 0:

            add_issue(
                "IIoT",
                "Power_Factor",
                "Power Factor Outside -1 to 1",
                count,
                "High"
            )

    if "Humidity_%" in df.columns:

        count = (
            (df["Humidity_%"] < 0) |
            (df["Humidity_%"] > 100)
        ).sum()

        if count > 0:

            add_issue(
                "IIoT",
                "Humidity_%",
                "Humidity Outside 0-100 Range",
                count,
                "Medium"
            )

    if "Timestamp" in df.columns:

        dates = pd.to_datetime(
            df["Timestamp"],
            errors="coerce"
        )

        count = (
            dates.isna() &
            df["Timestamp"].notna()
        ).sum()

        if count > 0:

            add_issue(
                "IIoT",
                "Timestamp",
                "Invalid Timestamp",
                count,
                "High"
            )

# =========================================================
# 9. MAINTENANCE QUALITY
# =========================================================

if "Maintenance" in data:

    df = data["Maintenance"]

    columns = [
        "Air temperature [K]",
        "Process temperature [K]",
        "Rotational speed [rpm]",
        "Torque [Nm]",
        "Tool wear [min]"
    ]

    for column in columns:

        if column in df.columns:

            count = (df[column] < 0).sum()

            if count > 0:

                add_issue(
                    "Maintenance",
                    column,
                    "Negative Value",
                    count,
                    "High"
                )

# =========================================================
# 10. BILLING QUALITY
# =========================================================

if "Billing" in data:

    df = data["Billing"]

    columns = [
        "Uses.Consumed",
        "Uses.Losses",
        "Uses.Total",
        "Revenue.Total",
        "Retail.Total.Revenue",
        "Retail.Total.Sales",
        "Retail.Total.Customers"
    ]

    for column in columns:

        if column in df.columns:

            count = (df[column] < 0).sum()

            if count > 0:

                add_issue(
                    "Billing",
                    column,
                    "Negative Billing/Usage Value",
                    count,
                    "High"
                )

    # Uses.Total consistency

    required = [
        "Uses.Total",
        "Uses.Consumed",
        "Uses.Losses",
        "Uses.Resale",
        "Uses.No Charge"
    ]

    if all(column in df.columns for column in required):

        calculated = (
            df["Uses.Consumed"].fillna(0) +
            df["Uses.Losses"].fillna(0) +
            df["Uses.Resale"].fillna(0) +
            df["Uses.No Charge"].fillna(0)
        )

        count = (
            abs(
                df["Uses.Total"].fillna(0)
                - calculated
            ) > 0.01
        ).sum()

        if count > 0:

            add_issue(
                "Billing",
                "Uses.Total",
                "Usage Total Inconsistency",
                count,
                "Medium"
            )

# =========================================================
# 11. OMS QUALITY
# =========================================================

if "OMS" in data:

    df = data["OMS"]

    if "Demand Loss (MW)" in df.columns:

        count = (
            df["Demand Loss (MW)"] < 0
        ).sum()

        if count > 0:

            add_issue(
                "OMS",
                "Demand Loss (MW)",
                "Negative Demand Loss",
                count,
                "High"
            )

    if "Number of Customers Affected" in df.columns:

        count = (
            df["Number of Customers Affected"] < 0
        ).sum()

        if count > 0:

            add_issue(
                "OMS",
                "Number of Customers Affected",
                "Negative Customer Count",
                count,
                "High"
            )

    for column in [
        "Date Event Began",
        "Date of Restoration"
    ]:

        if column in df.columns:

            dates = pd.to_datetime(
                df[column],
                errors="coerce"
            )

            count = (
                dates.isna() &
                df[column].notna()
            ).sum()

            if count > 0:

                add_issue(
                    "OMS",
                    column,
                    "Invalid Date",
                    count,
                    "Medium"
                )

# =========================================================
# 12. CREATE QUALITY REPORT
# =========================================================

quality_df = pd.DataFrame(
    quality,
    columns=[
        "Dataset",
        "Column",
        "Quality Issue",
        "Issue Count",
        "Severity"
    ]
)

quality_df.to_excel(
    "Data_Quality_Report.xlsx",
    index=False
)

# =========================================================
# 13. DISPLAY RESULTS
# =========================================================

print("\n" + "=" * 70)
print("DATA QUALITY REPORT")
print("=" * 70)

if quality_df.empty:
    print("No quality issues detected.")
else:
    print(quality_df.to_string(index=False))

print("\nData Quality completed.")
print("Created: Data_Quality_Report.xlsx")
```
